# Train a stand / sit / fall detection model

Trains YOLOv8 on the [stand-sit-fall](https://universe.roboflow.com/swook1015-ijop5/stand-sit-fall)
Roboflow dataset (8,340 images; the dataset's reported baseline is ~81%
mAP50, 85.2% precision, 74.6% recall - use that as a rough sanity check for
your own run, not a guarantee) and exports an ONNX file that drops straight
into `human-anomaly-detection-backend-main/best.onnx` - same input tensor
name (`images`, 640x640) and output shape (`output0`,
`[1, 4+numClasses, 8400]`) that `inference.js` already expects, generalized
to however many classes this dataset actually has (previously 2, now 3).

**Before running:** go to Runtime -> Change runtime type -> select **T4 GPU**.
Training on CPU will be extremely slow.

In [ ]:
!pip install ultralytics roboflow -q

## 1. Download the dataset from Roboflow

Workspace/project are filled in from the Universe URL you gave
(`swook1015-ijop5` / `stand-sit-fall`). Confirmed real class order by
downloading the export directly: **`0: fall, 1: sit, 2: stand`** - that's
what step 5 below will print again, and what `CLASS_NAMES` must match once
you deploy this model. I couldn't confirm your API key programmatically
though (Roboflow blocks unauthenticated lookups) - get that plus the exact
version number from the project page: **Versions** tab -> open the version
you want -> **Download Dataset** -> format **YOLOv8** -> "show download
code". Paste your API key and confirm/replace the version number below.

Use your **Private API Key** (Roboflow account Settings -> API Keys), not
the old public inference key that used to be hardcoded in the frontend -
that one's already been exposed publicly and shouldn't be reused for
anything new.

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "YOUR_PRIVATE_API_KEY"
WORKSPACE = "swook1015-ijop5"
PROJECT = "stand-sit-fall"
VERSION = 1  # confirm this against the project's Versions tab

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")
print(dataset.location)

## 2. Train

Starting from `yolov8n.pt` (the smallest YOLOv8 variant) on purpose - the
deployed backend runs on a memory/CPU-constrained free-tier host (see the
backend's git history for the OOM-crash and 30s-inference issues that had
to be worked around), so a bigger base model would make that worse, not
better. Adjust `epochs`/`batch` if you hit Colab's session time or GPU
memory limits.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    imgsz=640,
    epochs=100,
    patience=20,
    batch=16,
    project="stand-sit-fall",
    name="run1",
)

## 3. Check validation metrics before trusting this model

Compare against the dataset's reported baseline (~81% mAP50, 85.2%
precision, 74.6% recall) as a rough reference point.

In [ ]:
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("precision:", metrics.box.mp)
print("recall:", metrics.box.mr)

## 4. Export to ONNX

Do **not** add `nms=True` here - `inference.js` does its own confidence
thresholding and NMS on the raw output, matching the current model's
un-NMSed `[1, 6, 8400]` shape (which will become `[1, 7, 8400]` with 3
classes instead of 2 - the postprocessing code already handles this
generically via `CLASS_NAMES.length`, no code change needed). Adding
built-in NMS would change the output shape/format and break that code.

In [ ]:
model.export(format="onnx", imgsz=640, opset=12, simplify=True)

## 5. Get your real class names and order

**This matters.** The backend's `CLASS_NAMES` environment variable must list
these in the exact same order - index 0 here must be index 0 in
`CLASS_NAMES`, etc. Whatever this prints, that's the truth; don't assume
it's alphabetical or matches any particular order.

In [ ]:
import yaml

with open(f"{dataset.location}/data.yaml") as f:
    data_yaml = yaml.safe_load(f)
print("Class order:", data_yaml["names"])

## 6. Download the trained model

In [ ]:
from google.colab import files

files.download("stand-sit-fall/run1/weights/best.onnx")

## 7. Deploy the new model

1. Replace `human-anomaly-detection-backend-main/best.onnx` with the
   downloaded file.
2. Set the `CLASS_NAMES` environment variable to `fall,sit,stand` (confirmed
   real order - double check step 5's output actually matches before
   deploying, in case you used a different dataset version) - in both
   `render.yaml` (so it's correct on the next fresh deploy) and directly in
   the Render dashboard for the already-running service (env var changes
   there don't need a code push, just a restart).
3. This is also the moment to make the portfolio's project description
   accurate again - it currently says "detects falls... sitting,
   standing...", which was previously false advertising for a 2-class
   fall-only model but will now actually be true.
4. Commit and push `best.onnx` - Render will rebuild and redeploy
   automatically.
5. Re-verify with a real test image against `POST /analyze` once it's live,
   same way this was checked earlier in the project's history.